# AQR Momentum Strategies

**FINM 36700 Portfolio and Risk Management**

Case link: [AQR Momentum Strategies](https://markhendricks.github.io/finm-portfolio/case_studies/AQR%20Momentum%20Strategies.html)

Data: `data/momentum_data.xlsx`

# Section 2.1 - Is Momentum Still Profitable?

## 2.1 (a) Long-short momentum performance

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

path = "data/momentum_data.xlsx"
read = lambda s: pd.read_excel(path, sheet_name=s, index_col=0, parse_dates=True)
fac, ss = read("factors (excess returns)"), read("size_sorts (total returns)")
umd = read("momentum (excess returns)")["UMD"]

# UMD tab vs. the FF construction 1/2(BU + SU) - 1/2(BD + SD)
built = 0.5 * (ss["BIG HiPRIOR"] + ss["SMALL HiPRIOR"]) - 0.5 * (ss["BIG LoPRIOR"] + ss["SMALL LoPRIOR"])
print(f"UMD vs. size-sort construction: max abs difference {(umd - built).abs().max():.4f}")

D = pd.concat([umd.rename("MOM"), fac[["MKT", "HML"]]], axis=1).loc[:"2024"]
subsamples = {"1927-2024": ("1927", "2024"), "1927-1993": ("1927", "1993"),
              "1994-2008": ("1994", "2008"), "2009-2024": ("2009", "2024")}

def mom_stats(d):
    r = d["MOM"]
    return pd.Series({"mean": r.mean() * 12, "vol": r.std() * np.sqrt(12),
                      "Sharpe": r.mean() / r.std() * np.sqrt(12), "skewness": r.skew(),
                      "corr. to MKT": r.corr(d["MKT"]), "corr. to HML": r.corr(d["HML"]),
                      "t-stat (mean)": r.mean() / r.std() * np.sqrt(len(r)), "months": len(r)})

table1 = pd.DataFrame({k: mom_stats(D.loc[a:b]) for k, (a, b) in subsamples.items()}).T.rename_axis("Subsample")
table1["months"] = table1["months"].astype(int)
table1.style.format({"mean": "{:.2%}", "vol": "{:.2%}", "Sharpe": "{:.2f}", "skewness": "{:.2f}",
                     "corr. to MKT": "{:.2f}", "corr. to HML": "{:.2f}", "t-stat (mean)": "{:.2f}"})

UMD vs. size-sort construction: max abs difference 0.0001


,mean,vol,Sharpe,skewness,corr. to MKT,corr. to HML,t-stat (mean),months
Subsample,,,,,,,,
1927-2024,7.39%,16.27%,0.45,-3.07,-0.35,-0.41,4.49,1176
1927-1993,8.81%,16.05%,0.55,-3.91,-0.37,-0.50,4.49,804
1994-2008,10.27%,17.54%,0.59,-0.60,-0.24,-0.12,2.27,180
2009-2024,-1.29%,15.80%,-0.08,-2.69,-0.37,-0.32,-0.33,192


## 2.1 (b) Has momentum changed over time?

In [2]:
pre, post = D.loc["1927":"1993", "MOM"], D.loc["2009":"2024", "MOM"]
diff = 12 * (post.mean() - pre.mean())
se = 12 * np.sqrt(pre.var() / len(pre) + post.var() / len(post))
print(f"2009-2024 minus 1927-1993 mean: {diff:.2%} per year, t = {diff / se:.2f}")

2009-2024 minus 1927-1993 mean: -10.10% per year, t = -2.29


* After the publication in 1993 (1994–2008), momentum did not disappear. The mean rose to 10.27% (Sharpe 0.59) from 8.81% (Sharpe 0.55) in 1927–1993, and was still significant (t = 2.27).
* After the AQR case in 2008, the premium vanished. The mean in 2009–2024 is −1.29% (Sharpe −0.08, t = −0.33). That is 10.1 points a year below 1927–1993 (t = −2.29), so the drop is unlikely to be noise alone.
* Volatility is stable across all subsamples (15.8%–17.5%).
* Skew is negative in every period: −3.91 pre-1994, −0.60 in 1994–2008 and −2.69 since 2009.
* Correlations stay negative throughout. To MKT they are between −0.24 to −0.37, and to HML −0.12 to −0.50. The diversifying role is stable even when the mean is not.

## 2.1 (c) Is momentum an important piece of the ideal portfolio?

In [3]:
def tangency_sharpe(X):
    mu, cov = X.mean().values, X.cov().values
    return np.sqrt(12 * mu @ np.linalg.solve(cov, mu))

rows = {}
for k, (a, b) in subsamples.items():
    d = D.loc[a:b].copy()
    d["MOM (mean 0)"] = d["MOM"] - d["MOM"].mean()
    fit = sm.OLS(d["MOM"], sm.add_constant(d[["MKT", "HML"]])).fit()
    fit0 = sm.OLS(d["MOM (mean 0)"], sm.add_constant(d[["MKT", "HML"]])).fit()
    rows[k] = {"alpha": fit.params["const"] * 12, "t(alpha)": fit.tvalues["const"],
               "beta MKT": fit.params["MKT"], "beta HML": fit.params["HML"],
               "alpha, mean 0": fit0.params["const"] * 12, "t(alpha, mean 0)": fit0.tvalues["const"],
               "SR: MKT+HML": tangency_sharpe(d[["MKT", "HML"]]),
               "SR: + MOM": tangency_sharpe(d[["MKT", "HML", "MOM"]]),
               "SR: + MOM (mean 0)": tangency_sharpe(d[["MKT", "HML", "MOM (mean 0)"]])}

(pd.DataFrame(rows).T.rename_axis("Subsample")
   .style.format("{:.2f}").format("{:.2%}", subset=["alpha", "alpha, mean 0"]))

,alpha,t(alpha),beta MKT,beta HML,"alpha, mean 0","t(alpha, mean 0)",SR: MKT+HML,SR: + MOM,SR: + MOM (mean 0)
Subsample,,,,,,,,,
1927-2024,11.23%,7.73,-0.24,-0.46,3.84%,2.64,0.50,0.94,0.57
1927-1993,13.16%,7.90,-0.18,-0.54,4.34%,2.61,0.51,1.10,0.60
1994-2008,13.54%,3.10,-0.38,-0.40,3.26%,0.75,0.61,1.02,0.64
2009-2024,2.68%,0.73,-0.33,-0.35,3.97%,1.08,0.94,0.96,0.99


Alphas come from regressing MOM on MKT and HML. "Mean 0" re-runs everything with momentum's sample mean subtracted (momentum near zero due to transaction costs). SR is the annualized in-sample tangency Sharpe ratio of the listed assets.

The data supports AQR's that mmentum is an important piece of the ideal portfolio because momentum has positive mean excess returns and it has negative correlation with MKT and HML.

Momentum has large alpha relative to MKT and HML, momentum's full-sample alpha is 11.23% (t = 7.73), above its 7.39% mean. Its negative betas (−0.24 on MKT, −0.46 on HML) mean it pays off when the other two do poorly.
Also, adding momentum lifts the full-sample tangency Sharpe from 0.50 to 0.94.

Aftern with a zero mean after setting momentum's mean to zero (as if transaction costs consumed it all):

* Alpha stays positive at 3.84% (t = 2.64). This is pure hedging value, 0.24 × the market premium plus 0.46 × the value premium.
* The the tangency Sharpe still rises from 0.50 to 0.57, because momentum offsets losses in value and the market.

## Citations

Claude Opus 5.5 (Anthropic) was used to assist with the code, writing, and analysis in this notebook.